In [167]:
import os
from pathlib import Path
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from helpers.utils import get_prompt
import json
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [168]:
Path = r"D:\Kenpath\oan-evaluation"

In [169]:
os.chdir(Path)

In [170]:
from dotenv import load_dotenv
load_dotenv()


True

In [171]:
class Metric(BaseModel):
    metric_name: str = Field(..., description="The name of the metric to evaluate")
    score: int = Field(..., description="The score of the metric", ge=0, le=5)
    explanation: str = Field(..., description="The explanation of the score")

In [172]:
class EvaluationResult(BaseModel):
    """Result of the evaluation."""
    source_attribution: Metric = Field(..., description="Measures how well the response references appropriate, authoritative, and relevant sources. Higher scores reflect correct alignment between cited sources and the information presented.")
    grammatical_accuracy: Metric = Field(..., description="Evaluates the degree to which the response maintains grammatical correctness, valid word usage, accurate translations or transliterations, and proper sentence structure in Marathi.")
    factual_correctness: Metric = Field(..., description="Measures the accuracy of all factual information in the response, including details related to government schemes, mandi prices, fertilizers, pesticides, quantities, timelines, and agricultural guidelines, without introducing incorrect or misleading claims")
    actionability: Metric = Field(..., description="Evaluates the extent to which the response provides practical and implementable guidance, such as quantities, ratios, step-by-step instructions, timelines, or decision criteria that can be acted upon directly by the user.")
    completeness: Metric = Field(..., description="Measures how thoroughly the response addresses the user's query, including coverage of essential steps, conditions, data points, constraints, and safety considerations where relevant.")
    context_alignment: Metric = Field(..., description="Measures how closely the response stays aligned with the user's question and situational context, avoiding unnecessary or unrelated information, and introducing follow-up questions only when contextually justified.")
    information: Metric = Field(..., description="Evaluates whether the level of detail is suitable for the intended user, balancing clarity and precision without being overly technical or overly vague, and reflecting real-world agricultural practices.")
    safety_compliance: Metric = Field(..., description="Measures adherence to regulatory and safety norms by avoiding recommendations of banned, unsafe, or non-compliant agricultural inputs or practices, and by including appropriate safety precautions where applicable.")
    tool_usage: Metric = Field(..., description="Measures whether the response reflects correct selection and sequencing of tools or processes as required by the task, without omission, misuse, or unnecessary invocation.")
    retrieval_grounding: Metric = Field(..., description="Evaluates the extent to which the response is grounded in retrieved or provided documents and avoids introducing unsupported, speculative, or fabricated information.")

In [173]:
evaluation_agent = Agent(
    model='gpt-5',
    name="Evaluation Agent",
    instrument=True,
    output_type=EvaluationResult,
    retries=2,
    system_prompt=get_prompt('evaluation_system'),
)

Response-1

In [210]:
text = """User Query: माझ्या भागातील पुढील ५ दिवसांचा हवामान अंदाज काय आहे?
Language: Marathi
==================================================

Assistant called tool: fetch_agristack_data
Arguments: {}

Tool 'fetch_agristack_data' returned:
> Farmer Information (Agristack)
No farmer information found for the requested ID.
==================================================
==================================================

Final Response:
आपल्या भागासाठी हवामानाची माहिती मिळवण्यासाठी जिल्ह्याचे नाव सांगू शकता का? कृपया "महाराष्ट्रातील कोणत्या जिल्ह्यासाठी हवामान माहिती हवी आहे?" हे सांगा म्हणजे पुढील ५ दिवसांचा हवामान अंदाज देऊ शकतो."""

In [211]:
# import import_ipynb
from notebooks import *

In [212]:
eval_result = await evaluation_agent.run(text)

In [213]:
print(json.dumps(eval_result.output.model_dump(), indent=2, ensure_ascii=False))

{
  "source_attribution": {
    "metric_name": "source_attribution",
    "score": 0,
    "explanation": "No sources cited and no external data referenced; the response is a clarifying question without attribution."
  },
  "grammatical_accuracy": {
    "metric_name": "grammatical_accuracy",
    "score": 4,
    "explanation": "Marathi is clear and natural overall; minor stylistic issue (\"हवामान माहिती\" could be \"हवामानाची माहिती\")."
  },
  "factual_correctness": {
    "metric_name": "factual_correctness",
    "score": 5,
    "explanation": "No factual claims were made; asking for the district to provide a forecast is appropriate and correct."
  },
  "actionability": {
    "metric_name": "actionability",
    "score": 3,
    "explanation": "Provides a clear next step (share district name), but no actual forecast or detailed steps yet."
  },
  "completeness": {
    "metric_name": "completeness",
    "score": 1,
    "explanation": "Does not provide the requested 5-day forecast; only asks

In [214]:
text_str = text[0] if isinstance(text, list) else text

query = text_str.split("User Query:")[1].splitlines()[0].strip()
language = text_str.split("Language:")[1].splitlines()[0].strip()
response = text_str.split("Final Response:")[1].strip()

df_new = pd.DataFrame.from_dict(
    eval_result.output.model_dump(),
    orient="index"
).reset_index(drop=True)

df_new.insert(0, "query", query)
df_new.insert(1, "response", response)
df_new.insert(2, "language", language)


In [215]:
output_file = "evaluation_sheet.xlsx"

In [216]:
df_existing = pd.read_excel(output_file)

In [217]:
df_final = pd.concat([df_existing, df_new], ignore_index=True)


In [ ]:
df_final.to_excel(output_file, index=False)